# Домашнее задание 1

Шаблон блокнота обучения

## Импортирование необходимых библиотек

In [ ]:
from PIL import Image
from glob import glob
import time
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, Dataset, DataLoader
from torchsummary import summary
from torchvision import transforms as T
from tensorflow import summary as tfsummary
import pickle
from sklearn.metrics import classification_report
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt
import os

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Чтение тренировочной и тестовой выборки

Инструкция для скачивания и загрузки фотографий в Сolab находится в github

In [ ]:
# Размер изображения
heigth_width = 32

CLASSES = ['Торт', 'Ласточка', 'Кошка'] # Здесь требуется указать ваши классы

# Картинки - обучающие и тестовые
images = []
images_t = []
# Классы - обучающие и тестовые
classes = []
classes_t = []

#
# ВАШ КОД ЧТЕНИЯ ДАННЫХ
#

train_X = np.array(images)
train_y = np.array(classes)

test_X = np.array(images_t)
test_y = np.array(classes_t)

### Вывод примера изображения

In [ ]:
Image.fromarray(train_X[99]).resize((512,512))

## Создание Трансформа, Dataset и Pytorch DataLoader'a

In [ ]:
# TRANSFORM

transform = T.Compose([
    # ...
])

# # Вывод преобразованного изображения
# Image.fromarray((transform(torch.Tensor(train_X[50]).permute(2, 0, 1)/255.).\
#                  permute(1, 2, 0).numpy()*255.).astype(np.uint8)).\
#                  resize((256, 256))

In [ ]:
#
# Ваш Datase-класс, применяющий transform
#

class HW1_Dataset(Dataset):
  def __init__(self, X, y, transform=None, p=0.0):
    assert X.size(0) == y.size(0)

In [ ]:
batch_size = 32
dataloader = {}

for (X, y), part in zip([(train_X, train_y), (test_X, test_y)],
                        ['train', 'test']):
    tensor_x = torch.Tensor(X)
    tensor_y = F.one_hot(torch.Tensor(y).to(torch.int64),
                         num_classes=len(CLASSES))/1.

    # создание объекта датасета
    dataset = HW1_Dataset(tensor_x, tensor_y,
                          transform if part=='train' else None,
                          p=0.5)

    # создание экземпляра класса DataLoader
    dataloader[part] = DataLoader(dataset, batch_size=batch_size,
                                  shuffle=True
                                  # ... доп. параметры при необходимости ...
                                  )

dataloader

## Создание модели

In [ ]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super(Normalize, self).__init__()
        self.mean = torch.tensor(mean).to(device)
        self.std = torch.tensor(std).to(device)

    def forward(self, input):
        x = input / 255.0
        x = x - self.mean
        x = x / self.std
        return x.permute(0, 3, 1, 2) # nhwc -> nm

### На выбор: собственная или дообучение

In [ ]:
# СОБСТВЕННАЯ МОДЕЛЬ
HIDDEN_SIZE = 48

class HW1_MLP(nn.Module):
    def __init__(self, hidden_size=32, classes=100, mean=None, std=None):
        super(HW1_MLP, self).__init__()

        self.seq = nn.Sequential(
            # ...
        )

    def forward(self, input):
        x = self.norm(input)
        return self.seq(x)

model = HW1_MLP(
    hidden_size=HIDDEN_SIZE,
    classes=len(CLASSES),
    # mean=custom_mean,
    # std=custom_std
)
model.to(device)


# ПРЕДОБУЧЕННАЯ МОДЕЛЬ
model_loaded = torch.hub.load("chenyaofo/pytorch-cifar-models",
                              "cifar100_mobilenetv2_x0_5",
                              #'cifar100_resnet20',
                              pretrained=True)

model = nn.Sequential(
    Normalize([0.5356, 0.5012, 0.4595], [0.2022, 0.2025, 0.2077]),
    model_loaded
).to(device)

In [ ]:
# ДЛЯ ПРЕДОБУЧЕННОЙ: Заморозка весов

##Выбор функции потерь и оптимизатора градиентного спуска

In [ ]:
# Ваши функция потерь, оптимизатор и настройка
criterion = None
optimizer = None

## Обучение модели по эпохам

Для ускорения обучения перевести на `device`

In [ ]:
EPOCHS = 250

steps_per_epoch = len(dataloader['train'])
steps_per_epoch_val = len(dataloader['test'])

for epoch in range(EPOCHS): # проход по набору данных несколько раз
    running_loss = 0.0
    model.train()
    for i, batch in enumerate(dataloader['train'], 0):
        # получение одного минибатча; batch это двуэлементный список из [inputs, labels]
        inputs, labels = batch

        # очищение прошлых градиентов с прошлой итерации
        optimizer.zero_grad()

        # прямой + обратный проходы + оптимизация
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # для подсчёта статистик
        running_loss += loss.item()

    print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / steps_per_epoch:.3f}')
    running_loss = 0.0
    model.eval()
    with torch.no_grad(): # отключение автоматического дифференцирования
        for i, data in enumerate(dataloader['test'], 0):
            inputs, labels = data

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
    print(f'[{epoch + 1}, {i + 1:5d}] val loss: {running_loss / steps_per_epoch_val:.3f}')

print('Обучение закончено')

##Проверка качества модели по классам на обучающей и тестовой выборках

In [ ]:
for part in ['train', 'test']:
    y_pred = []
    y_true = []
    with torch.no_grad(): # отключение автоматического дифференцирования
        for i, data in enumerate(dataloader[part], 0):
            inputs, labels = data

            outputs = model(inputs).detach().numpy()
            y_pred.append(outputs)
            y_true.append(labels.numpy())
        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        print(part)
        print(classification_report(y_true.argmax(axis=-1), y_pred.argmax(axis=-1),
                                    digits=4, target_names=list(map(str, CLASSES))))
        print('-'*55)

## Сохранение модели в ONNX

In [ ]:
!pip install onnx onnxscript onnxruntime

In [ ]:
# Входной тензор для модели
x = torch.randn(1, 3, heigth_width, heigth_width, requires_grad=True).to(device)
torch_out = model(x)

# Экспорт модели
torch.onnx.export(model,               # модель
                  x,                   # входной тензор (или кортеж нескольких тензоров)
                  "cifar100_CNN.onnx", # куда сохранить (либо путь к файлу либо fileObject)
                  export_params=True,  # сохраняет веса обученных параметров внутри файла модели
                  opset_version=18,    # версия ONNX
                  dynamo=False,        # отключает новый torch.onnx.dynamo exporter
                  do_constant_folding=True,  # следует ли выполнять укорачивание констант для оптимизации
                  input_names = ['input'],   # имя входного слоя
                  output_names = ['output'],  # имя выходного слоя
                  dynamic_axes={'input' : {0 : 'batch_size'},    # динамичные оси, в данном случае только размер пакета
                                'output' : {0 : 'batch_size'}})